The below chunk pulls ACS data and saves it to a dataframe using pytidycensus. Make sure you have the necessary libraries installed.

In [4]:
import pandas as pd
import geopandas as gpd
import pytidycensus as tc
from dotenv import load_dotenv
import os
from pathlib import Path

In [8]:
# read census api key from .env
load_dotenv()
CENSUS_API_KEY = "e42f6b0ecfc65388715dc2caf3cfa9ae6b5a3c54"

if CENSUS_API_KEY is None:
    print("Error: API key not found. Make sure .env file exists with OPENWEATHER_API_KEY set.")
else:
    print("API key loaded successfully!")

API key loaded successfully!


In [ ]:
# Retrieve the 2024 ACS 5-year estimates for Allegheny County census tracts.
# Variable IDs are passed without E/M suffixes; pytidycensus requests estimates
# and margins of error, then the MOE columns are removed below.
acs_education_variables = {
    f"B15003_{code:03d}E": label
    for code, label in {
        1: "Population age 25 and over (education universe)",
        2: "No schooling completed",
        3: "Nursery school", 
        4: "Kindergarten",
        5: "1st grade",
        6: "2nd grade",
        7: "3rd grade",
        8: "4th grade",
        9: "5th grade",
        10: "6th grade",
        11: "7th grade",
        12: "8th grade",
        13: "9th grade",
        14: "10th grade",
        15: "11th grade",
        16: "12th grade, no diploma",
        17: "Regular high school diploma",
        18: "GED or alternative credential",
        19: "Some college, less than 1 year",
        20: "Some college, 1 or more years, no degree",
        21: "Associate's degree",
        22: "Bachelor's degree",
        23: "Master's degree",
        24: "Professional school degree",
        25: "Doctorate degree",
    }.items()
}

acs_variable_names = {
    "B01003_001E": "pop_total",
    "B02001_002E": "pop_white",
    "B02001_003E": "pop_black",
    "B02001_005E": "pop_asian",
    "B03003_003E": "pop_hispan",
    "B19013_001E": "hhinc",
    "B25077_001E": "home_val",
    "B25001_001E": "tot_housing",
    "B25002_002E": "occ_housing",
    "B25002_003E": "vac_housing",
    #"B25003_001E": "occ_by_tenure",
    "B25003_002E": "owners",
    "B25003_003E": "renters",
    **{
        f"{variable}": label
        for variable, label in acs_education_variables.items()
    },
    "GEOID": "geoid",
    "NAME": "name",
    "state": "statefips",
    "county": "countyfips",
    "tract": "tractfips",
}

acs_variables = [
    "B01003_001E",  # Total population
    "B02001_002E",  # White alone
    "B02001_003E",  # Black or African American alone
    "B02001_005E",  # Asian alone
    "B03003_003E",  # Hispanic or Latino (ethnicity)
    "B19013_001E",  # Median household income
    "B25077_001E",  # Median home value
    "B25001_001E",  # Total housing units (vacancy-rate denominator)
    "B25002_002E",  # Occupied housing units
    "B25002_003E",  # Vacant housing units
    #"B25003_001E",  # Total occupied units by tenure
    "B25003_002E",  # Owner occupied
    "B25003_003E",  # Renter occupied
    *acs_education_variables.keys(),  # Every detailed educational-attainment level
]

full_acs_df = pd.DataFrame()

# get_acs breaks due to the number of variables being extracted, 
# so a for loop must be used to extract only one variable at a time then merge it into the full DataFrame.
for var in acs_variables:
    acs_df = tc.get_acs(
        geography="tract",
        variables=[var],
        state="PA",
        county="Allegheny",
        year=2024,
        survey="acs5",
        geometry=True,
        output="wide",
        api_key=CENSUS_API_KEY,
    )
    if full_acs_df.empty:
        # First pass: keep the original geometry and all columns from this tract dataset
        full_acs_df = acs_df.copy()
    else:
        # Merge only the join key and the new variable to avoid duplicate geometry columns
        full_acs_df = full_acs_df.merge(
            acs_df[["GEOID", var]],
            on="GEOID",
            how="outer"
        )

# Drop all margins of error, retaining estimate values and tract geometry.
acs_moe_columns = [column for column in full_acs_df.columns if column.endswith("_moe")]
full_acs_df = full_acs_df.drop(columns=acs_moe_columns)

# Give estimates and geography columns readable names; keep GeoPandas geometry intact.
full_acs_df = full_acs_df.rename(columns=acs_variable_names)

# ACS tract summary tables provide monthly owner-cost ranges, not a mean mortgage
# payment or the mortgage principal-and-interest amount, so an average mortgage
# amount cannot be calculated reliably from these data and is omitted.
print(f"Loaded {len(full_acs_df):,} Allegheny County census tracts.")
full_acs_df.head()

Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting data from the 2020-2024 5-year ACS
Getting dat

,geoid,geometry,pop_total,statefips,countyfips,tractfips,name,pop_white,pop_black,pop_asian,...,"12th grade, no diploma",Regular high school diploma,GED or alternative credential,"Some college, less than 1 year","Some college, 1 or more years, no degree",Associate's degree,Bachelor's degree,Master's degree,Professional school degree,Doctorate degree
0,42003010301,"POLYGON ((-79.99462 40.4358, -79.99129 40.4348...",1855,42,003,010301,"Allegheny County, Pennsylvania",661,1029,9,...,0,478,322,164,138,50,27,19,0,0
1,42003010302,"POLYGON ((-79.99371 40.43892, -79.98646 40.438...",4754,42,003,010302,"Allegheny County, Pennsylvania",3688,542,186,...,18,114,17,48,24,44,196,106,7,12
2,42003020100,"POLYGON ((-80.01538 40.44308, -80.00932 40.444...",4600,42,003,020100,"Allegheny County, Pennsylvania",2899,493,597,...,0,305,0,74,142,245,841,690,354,120
3,42003020300,"POLYGON ((-79.99613 40.44818, -79.9924 40.4504...",1928,42,003,020300,"Allegheny County, Pennsylvania",1526,51,256,...,2,72,0,91,22,39,642,633,218,94
4,42003030500,"POLYGON ((-79.99228 40.44173, -79.99214 40.442...",2978,42,003,030500,"Allegheny County, Pennsylvania",670,1782,243,...,13,372,126,84,369,190,460,286,58,142


In [ ]:
from pathlib import Path

# write the full ACS DataFrame to a geojson file
output_dir = Path(r"")
output_dir.mkdir(parents=True, exist_ok=True)

full_acs_df.to_file(output_dir / "full_acs_df.geojson", driver="GeoJSON")